# YouTube Lecture RAG Search Engine

An end-to-end Retrieval-Augmented Generation system that indexes YouTube lecture transcripts,
lets you ask grounded questions with timestamped, clickable sources, and auto-generates
notes, MCQs, quizzes, and study guides.



## Cell 1 -- Install Required Packages

In [1]:
!pip install -q -U youtube-transcript-api
!pip install -q sentence-transformers faiss-cpu google-generativeai gradio
print("All packages installed successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 69.4 MB/s eta 0:00:00
All packages installed successfully.


## Cell 2 -- Imports & Gemini API Configuration

In [2]:
# ============================================================
# CELL 2: IMPORTS & GEMINI API CONFIGURATION
# ============================================================
import os
import re
import json
import time
import hashlib
import traceback
import numpy as np
import pandas as pd
from typing import List, Dict, Tuple, Optional

import faiss
from sentence_transformers import SentenceTransformer
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import (
    TranscriptsDisabled, NoTranscriptFound, VideoUnavailable
)
import google.generativeai as genai
import gradio as gr

# ---- Gemini API Key configuration ----
# Get a FREE API key from https://aistudio.google.com/app/apikey
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")

if not GEMINI_API_KEY:
    try:
        from google.colab import userdata
        GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    except Exception:
        pass

# Fall back to manual input if still not found
if not GEMINI_API_KEY:
    GEMINI_API_KEY = input("Enter your Gemini API Key: ").strip()

if not GEMINI_API_KEY:
    raise ValueError("A Gemini API key is required to run this notebook.")

genai.configure(api_key=GEMINI_API_KEY)

GEMINI_MODEL_NAME = "gemini-2.5-flash"
gemini_model = genai.GenerativeModel(GEMINI_MODEL_NAME)

print(f"Gemini configured successfully using model: {GEMINI_MODEL_NAME}")

WEBSHARE_PROXY_USERNAME = os.environ.get("WEBSHARE_PROXY_USERNAME", "")
WEBSHARE_PROXY_PASSWORD = os.environ.get("WEBSHARE_PROXY_PASSWORD", "")

if WEBSHARE_PROXY_USERNAME and WEBSHARE_PROXY_PASSWORD:
    from youtube_transcript_api.proxies import WebshareProxyConfig
    ytt_api = YouTubeTranscriptApi(
        proxy_config=WebshareProxyConfig(
            proxy_username=WEBSHARE_PROXY_USERNAME,
            proxy_password=WEBSHARE_PROXY_PASSWORD,
        )
    )
    print("YouTube Transcript API configured with a Webshare residential proxy.")
else:
    ytt_api = YouTubeTranscriptApi()
    print(
        "No proxy configured for the YouTube Transcript API. If video indexing fails "
        "with an IP-blocking error, set WEBSHARE_PROXY_USERNAME / WEBSHARE_PROXY_PASSWORD "
        "above (see https://www.webshare.io/) and re-run this cell."
    )

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Enter your Gemini API Key: AQ.Ab8RN6IFrHZklE5WlOioPME4vg9LfUDb03zeM4gNptGDM_FhLg
Gemini configured successfully using model: gemini-2.5-flash
No proxy configured for the YouTube Transcript API. If video indexing fails with an IP-blocking error, set WEBSHARE_PROXY_USERNAME / WEBSHARE_PROXY_PASSWORD above (see https://www.webshare.io/) and re-run this cell.


## Cell 3 -- YouTube Transcript Extraction (with Timestamps)

In [3]:
# ============================================================
# CELL 3: YOUTUBE TRANSCRIPT EXTRACTION WITH TIMESTAMPS
# ============================================================

class TranscriptExtractionError(Exception):
    """Raised whenever a transcript cannot be fetched for a given video."""
    pass

try:
    from youtube_transcript_api._errors import RequestBlocked
except ImportError:
    class RequestBlocked(Exception):
        pass

try:
    from youtube_transcript_api._errors import IpBlocked
except ImportError:
    class IpBlocked(RequestBlocked):
        pass

try:
    from youtube_transcript_api._errors import AgeRestricted
except ImportError:
    class AgeRestricted(Exception):
        pass

try:
    from youtube_transcript_api._errors import VideoUnplayable
except ImportError:
    class VideoUnplayable(Exception):
        pass

try:
    from youtube_transcript_api._errors import PoTokenRequired
except ImportError:
    class PoTokenRequired(Exception):
        pass


def extract_video_id(url: str) -> Optional[str]:
    """
    Extracts the 11-character YouTube video ID from a variety of URL formats,
    or returns the input unchanged if it already looks like a bare video ID.
    """
    patterns = [
        r"(?:youtube\.com/watch\?v=)([\w-]{11})",
        r"(?:youtu\.be/)([\w-]{11})",
        r"(?:youtube\.com/embed/)([\w-]{11})",
        r"(?:youtube\.com/shorts/)([\w-]{11})",
        r"(?:youtube\.com/live/)([\w-]{11})",
    ]
    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)
    # Fallback: maybe the user already pasted a bare video ID
    if re.fullmatch(r"[\w-]{11}", url.strip()):
        return url.strip()
    return None


def get_youtube_transcript(video_url: str) -> Tuple[List[Dict], str]:
    """
    Fetches the transcript for a YouTube video and returns it as a list of
    segments: [{"text": str, "start": float, "end": float}, ...]

    Uses the modern instance-based youtube-transcript-api (>=1.0) API:
      ytt_api.list(video_id) -> TranscriptList
      transcript.fetch()      -> FetchedTranscript
      fetched.to_raw_data()   -> list of {"text", "start", "duration"} dicts

    Handles: disabled transcripts, missing transcripts, unavailable/unplayable
    videos, IP-blocking by YouTube (common on cloud IPs like Colab), and PO-token
    requirements, with clear, actionable error messages for each case.
    """
    video_id = extract_video_id(video_url)
    if not video_id:
        raise TranscriptExtractionError(
            f"Could not extract a valid YouTube video ID from: {video_url}"
        )

    try:
        transcript_list = ytt_api.list(video_id)


        try:
            transcript = transcript_list.find_transcript(["en"])
        except NoTranscriptFound:
            available_langs = [t.language_code for t in transcript_list]
            if not available_langs:
                raise
            transcript = transcript_list.find_generated_transcript(available_langs)
            if transcript.language_code != "en" and transcript.is_translatable:
                transcript = transcript.translate("en")

        fetched_transcript = transcript.fetch()
        raw_segments = fetched_transcript.to_raw_data()

    except TranscriptsDisabled:
        raise TranscriptExtractionError(
            f"Transcripts are disabled by the uploader for this video: {video_url}"
        )
    except NoTranscriptFound:
        raise TranscriptExtractionError(
            f"No transcript is available (in any language) for this video: {video_url}"
        )
    except VideoUnavailable:
        raise TranscriptExtractionError(
            f"This video is unavailable, private, or has been removed: {video_url}"
        )
    except (IpBlocked, RequestBlocked):
        raise TranscriptExtractionError(
            "YouTube is blocking transcript requests from this server's IP address. "
            "This is very common on Google Colab and other cloud IPs, and is NOT a bug "
            "in this notebook. Fix: sign up for a residential proxy at https://www.webshare.io/ "
            "and set WEBSHARE_PROXY_USERNAME / WEBSHARE_PROXY_PASSWORD in Cell 2, then re-run "
            "Cell 2 and try indexing again."
        )
    except PoTokenRequired:
        raise TranscriptExtractionError(
            f"This video requires a YouTube PO token that this library cannot currently "
            f"provide ({video_url}). Please try a different video."
        )
    except (AgeRestricted, VideoUnplayable):
        raise TranscriptExtractionError(
            f"This video can't be accessed due to age restriction or playback restrictions: {video_url}"
        )
    except Exception as e:
        raise TranscriptExtractionError(
            f"Unexpected error while fetching the transcript: {str(e)}"
        )

    # Normalize into a clean list of segments with absolute start/end times
    segments = []
    for entry in raw_segments:
        start = float(entry.get("start", 0.0))
        duration = float(entry.get("duration", 0.0))
        text = entry.get("text", "").strip().replace("\n", " ")
        if text:
            segments.append({"text": text, "start": start, "end": start + duration})

    if not segments:
        raise TranscriptExtractionError(
            f"Transcript for {video_url} was empty after cleaning."
        )

    return segments, video_id


print("Transcript extraction functions ready.")

Transcript extraction functions ready.


## Cell 4 -- Transcript Chunking (Semantic, Timestamp-Preserving)

In [4]:
# ============================================================
# CELL 4: TRANSCRIPT CHUNKING FUNCTION
# ============================================================

def chunk_transcript(
    segments: List[Dict],
    video_url: str,
    video_id: str,
    max_words: int = 120,
    overlap_segments: int = 2
) -> List[Dict]:
    """
    Groups raw transcript segments into semantically coherent chunks of
    roughly `max_words` words, preserving accurate start/end timestamps
    for every chunk. A small sliding overlap (in segments) is kept between
    consecutive chunks so context isn't lost at chunk boundaries.

    Returns a list of dicts with keys:
      chunk_text, start_time, end_time, video_url, video_id
    """
    chunks: List[Dict] = []
    current_segs: List[Dict] = []
    current_words: List[str] = []
    word_count = 0

    def flush_chunk():
        if not current_segs:
            return
        chunk_text = " ".join(current_words).strip()
        chunks.append({
            "chunk_text": chunk_text,
            "start_time": current_segs[0]["start"],
            "end_time": current_segs[-1]["end"],
            "video_url": video_url,
            "video_id": video_id,
        })

    for seg in segments:
        seg_word_count = len(seg["text"].split())

        # If adding this segment would overflow the chunk, flush and start a
        # new chunk that carries a small overlap from the previous one.
        if word_count + seg_word_count > max_words and current_segs:
            flush_chunk()
            overlap = current_segs[-overlap_segments:] if overlap_segments > 0 else []
            current_segs = list(overlap)
            current_words = [s["text"] for s in overlap]
            word_count = sum(len(w.split()) for w in current_words)

        current_segs.append(seg)
        current_words.append(seg["text"])
        word_count += seg_word_count

    flush_chunk()  # flush the final partial chunk
    return chunks


print("Chunking function ready.")

Chunking function ready.


## Cell 5 -- Embedding Generation (Sentence Transformers, with Caching)

In [5]:
# ============================================================
# CELL 5: EMBEDDING GENERATION FUNCTION
# ============================================================

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"   # fast, high-quality, 384-dim
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
EMBEDDING_DIM = embedding_model.get_sentence_embedding_dimension()

# In-memory embedding cache keyed by a hash of the chunk text.
# Prevents recomputing embeddings for chunks/videos already processed.
_embedding_cache: Dict[str, np.ndarray] = {}


def _hash_text(text: str) -> str:
    """Stable hash used as a cache key for a piece of text."""
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def generate_embeddings(chunks: List[Dict], batch_size: int = 32) -> np.ndarray:
    """
    Generates L2-normalized embeddings for a list of chunk dicts.
    Only computes embeddings for chunks not already present in the cache,
    which keeps re-indexing and repeated runs fast and efficient.
    """
    texts = [c["chunk_text"] for c in chunks]
    hashes = [_hash_text(t) for t in texts]

    to_compute_idx = [i for i, h in enumerate(hashes) if h not in _embedding_cache]

    if to_compute_idx:
        to_compute_texts = [texts[i] for i in to_compute_idx]
        computed = embedding_model.encode(
            to_compute_texts,
            batch_size=batch_size,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,   # so inner product == cosine similarity
        )
        for idx, emb in zip(to_compute_idx, computed):
            _embedding_cache[hashes[idx]] = emb

    embeddings = np.vstack([_embedding_cache[h] for h in hashes]).astype("float32")
    return embeddings


print(f"Embedding model \'{EMBEDDING_MODEL_NAME}\' loaded. Dimension = {EMBEDDING_DIM}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model 'all-MiniLM-L6-v2' loaded. Dimension = 384


/tmp/ipykernel_9294/3451134421.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  EMBEDDING_DIM = embedding_model.get_sentence_embedding_dimension()


## Cell 6 -- FAISS Vector Store (Multi-Video Knowledge Base)

In [6]:
# ============================================================
# CELL 6: FAISS VECTOR STORE CREATION
# ============================================================
# A single shared FAISS index + metadata store holds chunks from ALL indexed
# videos, enabling multi-video indexing into one combined knowledge base.

faiss_index = faiss.IndexFlatIP(EMBEDDING_DIM)   # inner product = cosine (vectors are normalized)
chunk_metadata_store: List[Dict] = []            # parallel list: metadata for each vector in faiss_index
indexed_video_ids: set = set()                   # tracks videos already indexed (avoids re-indexing)


def add_chunks_to_index(chunks: List[Dict], embeddings: np.ndarray) -> None:
    """Adds a batch of chunk embeddings + metadata to the shared vector store."""
    global faiss_index, chunk_metadata_store
    faiss_index.add(embeddings)
    chunk_metadata_store.extend(chunks)


def index_youtube_video(video_url: str) -> str:
    """
    Full indexing pipeline for a single video:
    transcript -> chunks -> embeddings -> FAISS.
    Returns a user-friendly status message. Never raises.
    """
    try:
        video_id = extract_video_id(video_url)
        if not video_id:
            return f"Invalid YouTube URL: {video_url}"

        if video_id in indexed_video_ids:
            return f"Video already indexed, skipping re-indexing: {video_url}"

        segments, video_id = get_youtube_transcript(video_url)
        chunks = chunk_transcript(segments, video_url, video_id)
        embeddings = generate_embeddings(chunks)
        add_chunks_to_index(chunks, embeddings)
        indexed_video_ids.add(video_id)

        return f"Indexed {len(chunks)} chunks ({len(segments)} raw segments) from video {video_id}."

    except TranscriptExtractionError as e:
        return f"Warning: {str(e)}"
    except Exception as e:
        return f"Unexpected error while indexing video: {str(e)}"


def reset_knowledge_base() -> str:
    """Clears the entire vector store (all indexed videos)."""
    global faiss_index, chunk_metadata_store, indexed_video_ids
    faiss_index = faiss.IndexFlatIP(EMBEDDING_DIM)
    chunk_metadata_store = []
    indexed_video_ids = set()
    return "Knowledge base has been reset."


print("FAISS vector store initialized.")

FAISS vector store initialized.


## Cell 7 -- Retriever (Top-K Semantic Search + Deduplication)

In [7]:
# ============================================================
# CELL 7: RETRIEVER IMPLEMENTATION
# ============================================================

def retrieve_relevant_chunks(query: str, top_k: int = 5) -> List[Dict]:
    """
    Embeds the query, performs a semantic similarity search over the shared
    FAISS index, and returns the top-k UNIQUE chunks (deduplicated by
    normalized text) sorted by relevance score.
    """
    if faiss_index.ntotal == 0:
        return []

    query_embedding = embedding_model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True
    ).astype("float32")

    # Over-fetch so that after deduplication we can still return top_k results
    search_k = min(top_k * 3, faiss_index.ntotal)
    scores, indices = faiss_index.search(query_embedding, search_k)

    seen_texts = set()
    results: List[Dict] = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        chunk = chunk_metadata_store[idx]
        normalized = chunk["chunk_text"].strip().lower()
        if normalized in seen_texts:
            continue
        seen_texts.add(normalized)

        result = dict(chunk)
        result["score"] = float(score)
        results.append(result)

        if len(results) >= top_k:
            break

    return results


print("Retriever ready.")

Retriever ready.


## Cell 8 -- RAG Pipeline (Q&A with Conversation Memory + Study-Tool Generators)\n\nThis cell contains the core RAG answer pipeline plus the map-reduce content generators used by the Notes / MCQ / Quiz / Study Guide features, since they all share the same LLM-calling and prompt-grounding infrastructure.

In [8]:
# ============================================================
# CELL 8: RAG PIPELINE IMPLEMENTATION
# ============================================================

conversation_history: List[Dict] = []   # in-memory chat memory: [{"role": "user"/"assistant", "content": str}]


def format_timestamp(seconds: float) -> str:
    """Formats seconds as HH:MM:SS (or MM:SS if under an hour)."""
    seconds = int(max(0, seconds))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}" if h > 0 else f"{m:02d}:{s:02d}"


try:
    from google.api_core.exceptions import ResourceExhausted, TooManyRequests
except ImportError:
    class ResourceExhausted(Exception):
        pass
    class TooManyRequests(ResourceExhausted):
        pass


def call_gemini(prompt: str, max_retries: int = 3) -> str:
    """
    Thin wrapper around the Gemini API call with error handling and automatic
    retry-with-backoff on rate-limit (429) errors -- important because the
    Notes/MCQ/Quiz/Study Guide tools fire several calls back-to-back and can
    easily trip the free tier's requests-per-minute limit.
    Never raises: always returns a string, and always prints the full
    traceback to the Colab console on failure so the real cause is visible
    even though the UI only shows a short message.
    """
    for attempt in range(max_retries + 1):
        try:
            response = gemini_model.generate_content(prompt)

            # response.text raises if Gemini returned no valid candidate (e.g. the
            # prompt or output was blocked by safety filters) -- handle that explicitly
            # instead of letting it bubble up as a generic exception.
            if not getattr(response, "candidates", None):
                reason = getattr(getattr(response, "prompt_feedback", None), "block_reason", "unknown")
                return f"Gemini did not return a response (blocked or empty). Reason: {reason}"

            text = (response.text or "").strip()
            return text or "Gemini returned an empty response."

        except (ResourceExhausted, TooManyRequests) as e:
            if attempt < max_retries:
                wait_seconds = 15 * (attempt + 1)
                print(
                    f"Gemini rate limit hit (attempt {attempt + 1}/{max_retries}). "
                    f"Waiting {wait_seconds}s before retrying..."
                )
                time.sleep(wait_seconds)
                continue
            print("=== Gemini API call failed after all retries -- full traceback below ===")
            traceback.print_exc()
            return (
                f"Gemini rate limit exceeded after {max_retries} retries: {str(e)}. "
                "Your free-tier quota is likely exhausted for now -- wait a minute and "
                "try again, or switch GEMINI_MODEL_NAME to 'gemini-2.5-flash-lite' in Cell 2."
            )
        except Exception as e:
            print("=== Gemini API call failed -- full traceback below ===")
            traceback.print_exc()
            return f"Error calling Gemini API: {str(e)}"


def build_qa_prompt(query: str, contexts: List[Dict], history: List[Dict]) -> str:
    """
    Builds a grounded prompt that forces the LLM to answer using ONLY the
    retrieved transcript excerpts, minimizing hallucination.
    """
    context_lines = []
    for i, c in enumerate(contexts):
        start_ts = format_timestamp(c["start_time"])
        end_ts = format_timestamp(c["end_time"])
        context_lines.append(f"[Chunk {i+1} | {start_ts}-{end_ts}]\n{c['chunk_text']}")
    context_block = "\n\n".join(context_lines)

    history_block = ""
    if history:
        recent_turns = history[-6:]  # last 3 user/assistant exchanges
        history_lines = [f"{t['role'].upper()}: {t['content']}" for t in recent_turns]
        history_block = "\n".join(history_lines)

    return f"""You are an AI assistant answering questions about YouTube lecture videos.
Use ONLY the transcript excerpts provided below as your source of truth. Do NOT use outside
knowledge. If the answer is not contained in the excerpts, clearly say you don't know based
on the available transcript.

Conversation so far:
{history_block if history_block else "(no previous turns)"}

Transcript excerpts:
{context_block}

Question: {query}

Instructions:
- Give a clear, well-structured answer grounded strictly in the excerpts above.
- Reference the chunk numbers (e.g. "[Chunk 2]") that support each claim.
- If multiple excerpts disagree or the topic isn't covered, say so explicitly.
"""


def rag_answer(query: str, top_k: int = 5) -> Dict:
    """
    Full RAG pipeline: retrieve -> build grounded prompt -> call LLM -> return
    answer + source chunks (for source attribution). Updates conversation memory.
    """
    if not query or not query.strip():
        return {"answer": "Please enter a question.", "sources": []}

    if faiss_index.ntotal == 0:
        return {"answer": "No videos indexed yet. Please index a YouTube video first.", "sources": []}

    try:
        contexts = retrieve_relevant_chunks(query, top_k=top_k)
        if not contexts:
            return {"answer": "Could not retrieve any relevant transcript chunks for this question.", "sources": []}

        prompt = build_qa_prompt(query, contexts, conversation_history)
        answer = call_gemini(prompt)

        conversation_history.append({"role": "user", "content": query})
        conversation_history.append({"role": "assistant", "content": answer})

        return {"answer": answer, "sources": contexts}

    except Exception as e:
        print("=== rag_answer failed -- full traceback below ===")
        traceback.print_exc()
        return {"answer": f"Unexpected error generating answer: {str(e)}", "sources": []}


def reset_conversation() -> str:
    """Clears follow-up conversation memory (does not affect the knowledge base)."""
    global conversation_history
    conversation_history = []
    return "Conversation memory cleared."


# ------------------------------------------------------------
# Study-tool generators: Notes, MCQs, Quiz, Study Guide.
# Uses a map-reduce strategy so ANY length of transcript (single video or many)
# can be summarized efficiently without exceeding LLM context limits.
# ------------------------------------------------------------

def _batch_chunks_by_chars(chunks: List[Dict], max_chars: int = 8000) -> List[List[Dict]]:
    """Groups chunks into batches whose combined text stays under max_chars."""
    batches, current, current_len = [], [], 0
    for c in chunks:
        text_len = len(c["chunk_text"])
        if current_len + text_len > max_chars and current:
            batches.append(current)
            current, current_len = [], 0
        current.append(c)
        current_len += text_len
    if current:
        batches.append(current)
    return batches


def _map_reduce_generate(map_instruction: str, reduce_instruction: str) -> str:
    """
    MAP step: run the instruction over every batch of the indexed transcript(s).
    REDUCE step: merge all partial outputs into one final, deduplicated result.
    This keeps generation efficient and grounded even for very long lectures
    or multiple indexed videos.
    """
    chunks = chunk_metadata_store
    if not chunks:
        return "No videos indexed yet. Please index at least one YouTube video first."

    batches = _batch_chunks_by_chars(chunks)
    partial_outputs = []
    for i, batch in enumerate(batches):
        text_block = "\n\n".join(c["chunk_text"] for c in batch)
        prompt = f"{map_instruction}\n\nTranscript segment {i+1}/{len(batches)}:\n{text_block}"
        partial_outputs.append(call_gemini(prompt))

    if len(partial_outputs) == 1:
        return partial_outputs[0]

    combined = "\n\n---\n\n".join(partial_outputs)
    reduce_prompt = f"{reduce_instruction}\n\nPartial results to merge:\n\n{combined}"
    return call_gemini(reduce_prompt)


def generate_lecture_notes() -> str:
    return _map_reduce_generate(
        map_instruction=(
            "Summarize the following lecture transcript segment into concise, well-organized "
            "study notes with headings and bullet points. Use only information present in the text."
        ),
        reduce_instruction=(
            "Merge the following partial lecture notes into one cohesive, non-repetitive, "
            "well-structured set of notes with clear headings and bullet points."
        ),
    )


def generate_mcqs(num_questions: int = 10) -> str:
    return _map_reduce_generate(
        map_instruction=(
            "Based on the following lecture transcript segment, write multiple-choice questions "
            "(4 options each, clearly mark the correct option) that test understanding of key "
            "concepts covered in this segment."
        ),
        reduce_instruction=(
            f"Merge and deduplicate the following partial sets of multiple-choice questions into "
            f"a single final numbered list of about {num_questions} high-quality MCQs, each with "
            f"4 options and the correct answer clearly marked."
        ),
    )


def generate_quiz(num_questions: int = 8) -> str:
    return _map_reduce_generate(
        map_instruction=(
            "Based on the following lecture transcript segment, write short-answer quiz questions "
            "together with their correct answers, testing understanding of key concepts."
        ),
        reduce_instruction=(
            f"Merge and deduplicate the following partial quizzes into a single final numbered list "
            f"of about {num_questions} question-answer pairs, formatted as \'Q: ... / A: ...\'."
        ),
    )


def generate_study_guide() -> str:
    return _map_reduce_generate(
        map_instruction=(
            "From the following lecture transcript segment, extract important concepts, formulas, "
            "definitions, and key takeaways."
        ),
        reduce_instruction=(
            "Merge the following partial extracts into one organized study guide with clear "
            "sections: \'Key Concepts\', \'Definitions\', \'Formulas\' (if any), and \'Key Takeaways\'."
        ),
    )


print("RAG pipeline and study-tool generators ready.")

RAG pipeline and study-tool generators ready.


## Cell 9 -- Timestamp to Clickable YouTube Link + Source Formatting

In [9]:
# ============================================================
# CELL 9: TIMESTAMP-TO-CLICKABLE-YOUTUBE-LINK FUNCTION
# ============================================================

def make_youtube_timestamp_link(video_url: str, start_time: float) -> str:
    """
    Builds a YouTube URL that jumps directly to `start_time` seconds,
    e.g. https://www.youtube.com/watch?v=VIDEOID&t=125s
    """
    video_id = extract_video_id(video_url)
    seconds = int(max(0, start_time))
    if video_id:
        return f"https://www.youtube.com/watch?v={video_id}&t={seconds}s"
    return video_url  # fallback: original URL if ID extraction fails


def format_sources_markdown(sources: List[Dict]) -> str:
    """
    Formats retrieved source chunks as Markdown for display in the UI:
    each source shows its timestamp range, a clickable jump-to-moment link,
    and a trimmed excerpt of the transcript text (source attribution).
    """
    if not sources:
        return "_No sources available._"

    blocks = []
    for i, c in enumerate(sources, start=1):
        link = make_youtube_timestamp_link(c["video_url"], c["start_time"])
        start_ts = format_timestamp(c["start_time"])
        end_ts = format_timestamp(c["end_time"])
        ts_range = f"{start_ts} - {end_ts}"
        excerpt = c["chunk_text"]
        if len(excerpt) > 320:
            excerpt = excerpt[:320].rsplit(" ", 1)[0] + "..."
        score = c.get("score")
        score_str = f" (relevance: {score:.2f})" if score is not None else ""
        blocks.append(
            f"**[{i}] Time {ts_range}**{score_str}  [Jump to this moment]({link})\n\n> {excerpt}\n"
        )
    return "\n---\n".join(blocks)


print("Timestamp linking + source formatting ready.")

Timestamp linking + source formatting ready.


## Cell 10 -- Gradio UI

In [10]:
# ============================================================
# CELL 10: GRADIO UI
# ============================================================

def get_indexed_videos_markdown() -> str:
    """Small helper: shows which videos are currently in the knowledge base."""
    if not indexed_video_ids:
        return "_No videos indexed yet._"
    lines = [f"- `{vid}` -- https://www.youtube.com/watch?v={vid}" for vid in sorted(indexed_video_ids)]
    return "**Indexed videos (" + str(len(indexed_video_ids)) + "):**\n" + "\n".join(lines)


def ui_index_video(url: str):
    try:
        if not url or not url.strip():
            return "Please enter a YouTube URL.", get_indexed_videos_markdown()
        status = index_youtube_video(url.strip())
        return status, get_indexed_videos_markdown()
    except Exception as e:
        print("=== ui_index_video failed -- full traceback below ===")
        traceback.print_exc()
        return f"Unexpected UI error: {str(e)}", get_indexed_videos_markdown()


def ui_reset_kb():
    try:
        status = reset_knowledge_base()
        reset_conversation()
        return status, get_indexed_videos_markdown()
    except Exception as e:
        print("=== ui_reset_kb failed -- full traceback below ===")
        traceback.print_exc()
        return f"Unexpected UI error: {str(e)}", get_indexed_videos_markdown()


def ui_ask_question(question: str, top_k: float):
    try:
        result = rag_answer(question, top_k=int(top_k))
        return result["answer"], format_sources_markdown(result["sources"])
    except Exception as e:
        print("=== ui_ask_question failed -- full traceback below ===")
        traceback.print_exc()
        return f"Unexpected UI error: {str(e)}", "_No sources available._"


def ui_reset_convo():
    try:
        return reset_conversation()
    except Exception as e:
        print("=== ui_reset_convo failed -- full traceback below ===")
        traceback.print_exc()
        return f"Unexpected UI error: {str(e)}"


def ui_generate_notes():
    try:
        return generate_lecture_notes()
    except Exception as e:
        print("=== ui_generate_notes failed -- full traceback below ===")
        traceback.print_exc()
        return f"Unexpected UI error: {str(e)}"


def ui_generate_mcqs():
    try:
        return generate_mcqs()
    except Exception as e:
        print("=== ui_generate_mcqs failed -- full traceback below ===")
        traceback.print_exc()
        return f"Unexpected UI error: {str(e)}"


def ui_generate_quiz():
    try:
        return generate_quiz()
    except Exception as e:
        print("=== ui_generate_quiz failed -- full traceback below ===")
        traceback.print_exc()
        return f"Unexpected UI error: {str(e)}"


def ui_generate_study_guide():
    try:
        return generate_study_guide()
    except Exception as e:
        print("=== ui_generate_study_guide failed -- full traceback below ===")
        traceback.print_exc()
        return f"Unexpected UI error: {str(e)}"


# ---------------------------- Build the interface ----------------------------

with gr.Blocks(title="YouTube Lecture RAG Search Engine", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "# YouTube Lecture RAG Search Engine\n"
        "Index one or more YouTube lecture videos, ask grounded questions with clickable "
        "timestamped sources, and auto-generate notes, MCQs, quizzes, and study guides."
    )

    with gr.Tab("Index Videos"):
        gr.Markdown("Paste a YouTube lecture URL and index it. You can add **multiple videos** -- they all share one knowledge base.")
        with gr.Row():
            url_input = gr.Textbox(label="YouTube Video URL", placeholder="https://www.youtube.com/watch?v=...", scale=4)
            index_btn = gr.Button("Index Video", variant="primary", scale=1)
        index_status = gr.Markdown()
        indexed_list = gr.Markdown(get_indexed_videos_markdown())
        reset_kb_btn = gr.Button("Reset Knowledge Base", variant="stop")

        index_btn.click(ui_index_video, inputs=[url_input], outputs=[index_status, indexed_list])
        reset_kb_btn.click(ui_reset_kb, outputs=[index_status, indexed_list])

    with gr.Tab("Ask Questions"):
        gr.Markdown("Ask anything about the indexed videos. Follow-up questions are supported (conversation memory).")
        question_input = gr.Textbox(label="Your question", placeholder="What does the lecturer say about...?", lines=2)
        top_k_slider = gr.Slider(1, 10, value=5, step=1, label="Number of transcript chunks to retrieve (top-k)")
        ask_btn = gr.Button("Ask", variant="primary")

        gr.Markdown("### Answer")
        answer_output = gr.Markdown()
        gr.Markdown("### Sources & Timestamps")
        sources_output = gr.Markdown()

        clear_convo_btn = gr.Button("Clear Conversation Memory")
        convo_status = gr.Markdown()

        ask_btn.click(ui_ask_question, inputs=[question_input, top_k_slider], outputs=[answer_output, sources_output])
        clear_convo_btn.click(ui_reset_convo, outputs=[convo_status])

    with gr.Tab("Study Tools"):
        gr.Markdown("Generate study materials from **all currently indexed videos**.")
        with gr.Row():
            notes_btn = gr.Button("Lecture Notes")
            mcq_btn = gr.Button("MCQs")
            quiz_btn = gr.Button("Short-Answer Quiz")
            guide_btn = gr.Button("Study Guide")
        study_output = gr.Markdown()

        notes_btn.click(ui_generate_notes, outputs=[study_output])
        mcq_btn.click(ui_generate_mcqs, outputs=[study_output])
        quiz_btn.click(ui_generate_quiz, outputs=[study_output])
        guide_btn.click(ui_generate_study_guide, outputs=[study_output])

    gr.Markdown(
        "---\n_Answers and study materials are grounded strictly in indexed transcript content. "
        "Always verify important details against the original video._"
    )

print("Gradio UI defined. Run the next cell to launch it.")

/tmp/ipykernel_9294/3744856472.py:99: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="YouTube Lecture RAG Search Engine", theme=gr.themes.Soft()) as demo:


Gradio UI defined. Run the next cell to launch it.


## Cell 11 -- Launch Application

In [11]:
# ============================================================
# CELL 11: LAUNCH APPLICATION
# ============================================================

demo.queue().launch(share=True, debug=True, show_error=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e446905fdf43268f09.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://e446905fdf43268f09.gradio.live
